# Classification of Legal Text: live demo
    **Team 9:** Sai Sathvik Gullipalli (PES2UG24CS430) and Priyanka Mallikarjuna Reddy (PES2UG24CS378).

    Classify an English legal opinion and find similar opinions using saved models.
    This notebook does **not** retrain models. Use `team9-project.zip` and
    `team9-model-bundle.zip` from the private release, or the complete offline ZIP.
    Load only the team's trusted model bundle: joblib files contain executable Python objects.

In [1]:
import os, sys, json, subprocess, zipfile
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

def extract_trusted_zip(path, target):
    target = Path(target).resolve()
    with zipfile.ZipFile(path) as z:
        for member in z.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Archive contains an unsafe path")
        z.extractall(target)

def find_project():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent, parent / "legal_text_classification"]:
            if (candidate / "src" / "legal_text").is_dir():
                return candidate.resolve()
    return None

ROOT = find_project()
if ROOT is None and IS_COLAB:
    print("Upload team9-project.zip from the private GitHub release.")
    from google.colab import files
    for filename in files.upload():
        if filename.endswith(".zip"):
            extract_trusted_zip(filename, Path.cwd())
    ROOT = find_project()
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the extracted legal_text_classification project.")
os.chdir(ROOT)
if IS_COLAB:
    # Run this setup cell before importing NumPy/scikit-learn/gensim. If those
    # libraries were previously imported, restart the session after installation.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".build" / "matplotlib"))
print("Project:", ROOT)
print("Python:", sys.version.split()[0])

Project: /Users/admin/Documents/ChatGPT/meth/legal_text_classification
Python: 3.11.9


In [2]:
if not (ROOT / "artifacts/manifest.json").exists():
    if IS_COLAB:
        print("Upload team9-model-bundle.zip.")
        from google.colab import files
        for filename in files.upload():
            if filename.endswith(".zip"):
                extract_trusted_zip(filename, ROOT)
    else:
        raise FileNotFoundError("Extract team9-model-bundle.zip here, then rerun this cell.")
from legal_text.inference import load, predict, retrieve_similar
demo = load(ROOT / "artifacts")
print("Loaded models:", list(demo.models))
print("Validation-selected model:", demo.manifest["winner"])

Loaded models: ['tfidf', 'lda', 'doc2vec']
Validation-selected model: tfidf


## Saved inference example
    This cell provides a static backup if the widget interface is unavailable.
    Example passages come from the held-out test partition and may be truncated
    for display. The reported full-test metrics come from complete opinions.

In [3]:
import pandas as pd
from IPython.display import display
example = next(e for e in demo.examples if e["true_category"] == "Criminal Procedure")
print("Case:", example["case_name"])
print("True category:", example["true_category"])
for model in demo.models:
    result = predict(example["text"], model)
    print(model, "predicted:", result["category"])
    display(pd.DataFrame(result["top_three"]))
neighbors = retrieve_similar(example["text"], 5)
display(pd.DataFrame(neighbors)[["case_name", "decision_date", "category", "similarity"]])

Case: CARTER v. ILLINOIS
True category: Criminal Procedure
tfidf predicted: Criminal Procedure


,label,category,probability
0,1,Criminal Procedure,0.921634
1,2,Civil Rights,0.029259
2,4,Due Process,0.024848


lda predicted: Criminal Procedure


,label,category,probability
0,1,Criminal Procedure,0.957287
1,4,Due Process,0.019953
2,2,Civil Rights,0.016126


doc2vec predicted: Criminal Procedure


,label,category,probability
0,1,Criminal Procedure,0.836903
1,2,Civil Rights,0.086797
2,9,Judicial Power,0.051149


,case_name,decision_date,category,similarity
0,"MARINO v. RAGEN, WARDEN",1947-12-22,Criminal Procedure,0.655512
1,"MCNEAL v. CULVER, STATE PRISON CUSTODIAN",1961-01-23,Criminal Procedure,0.641854
2,UVEGES v. PENNSYLVANIA,1948-12-13,Criminal Procedure,0.629258
3,"GRYGER v. BURKE, WARDEN",1948-06-14,Criminal Procedure,0.577582
4,"BROWN v. ALLEN, WARDEN",1953-02-09,Criminal Procedure,0.570805


## Interactive demonstration
    Choose an example or paste a passage, select a model, then press the prediction
    button. TF-IDF also displays influential terms. Probabilities are uncalibrated.
    Short passages are less representative of the full opinions used for training.

In [4]:
from legal_text.widgets import launch
interactive_demo = launch(ROOT / "artifacts")

## Results and limitations
    Model probabilities measure model preference, not verified legal correctness.
    Similarity scores measure textual resemblance and do not establish binding precedent.
    The corpus covers historical US Supreme Court opinions, rather than Indian law
    or all types of legal documents.

In [5]:
if (ROOT / "results/metrics.json").exists():
    display(pd.DataFrame(json.loads((ROOT / "results/metrics.json").read_text())).T)
else:
    display(pd.DataFrame(json.loads((ROOT / "artifacts/metrics.json").read_text())).T)

,accuracy,macro_f1,weighted_f1,test_ms_per_document,train_seconds
majority,0.223881,0.028143,0.081908,NaN,NaN
tfidf,0.818507,0.718577,0.814364,7.142138,146.682140
lda,0.687761,0.522102,0.677283,5.103456,84.032056
doc2vec,0.734925,0.642808,0.729550,105.072569,1568.336844
